In [1]:
from pydantic import BaseModel, Field

class Pokemon(BaseModel):
    name:str = Field(description= "The name of the pokemon")
    type:str = Field(description= "What type of pokemon is it")
    rarity:str = Field(description= "Is the pokemon common to find or rare to find")
    gen:str = Field(description= "What generation does the pokemon belongs to")

In [2]:
from langchain_groq import ChatGroq
model = ChatGroq(
    model= "llama-3.1-8b-instant",
    temperature= 1
)

In [3]:
structured_model = model.with_structured_output(Pokemon)

In [4]:
model.invoke("What is Pikachu?")

AIMessage(content='Pikachu is a fictional creature and one of the most iconic characters from the popular media franchise "Pokémon." It is a small, rodent-like creature with electric powers. Pikachu is typically depicted as being yellow with electric spikes on its cheeks and red circles on its cheeks that can store electricity.\n\nAccording to the Pokémon universe, Pikachu can store electricity in its cheeks and release it through the circles. This ability allows it to defend itself or attack other Pokémon. Pikachu is often considered the mascot of the Pokémon franchise, being a popular character for both Pokémon fans and non-fans alike.\n\nPikachu first appeared in the first-generation Pokémon games, "Pokémon Red and Green," which were released in Japan in 1996. These games later became "Pokémon Red and Blue" for international releases.\n\nPikachu gained massive popularity worldwide due to its appearance in the "Pokémon" anime series, which was released in 1997. The character\'s adora

In [6]:
structured_model.batch(["What is Pikachu?",
                         "What is Lugia?"])

[Pokemon(name='Pikachu', type='Electric', rarity='common', gen='First generation'),
 Pokemon(name='Lugia', type='psychic/flying', rarity='rare', gen='2')]

In [ ]:
class DoAshHave(BaseModel):
    have: bool = Field(description= "Does Ask Ketchum have the pokemon")
    generation: str = Field(description= "When deos the Ash Ketchum captures(if he captured one).")
class OwnPokemon(BaseModel):
    name:str = Field(description= "The name of the pokemon")
    type:str = Field(description= "What type of pokemon is it")
    gen:str = Field(description= "What generation does the pokemon belongs to")
    Availabilty:DoAshHave = Field(description= "Does Ash has this pokemon")

In [17]:
nested_structured_model = model.with_structured_output(OwnPokemon)

In [25]:
nested_structured_model.invoke("""
Provide the following information:

- name
- type
- generation
- Availability MUST be an array of objects.
""")

OwnPokemon(name='Pikachu', type='Electric', gen='Generation 1', Availabilty=[DoAshHave(have=True, generation='Generation 1')])

In [27]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

Using TypeDict to verify the types, no need of Pydantic to validate at the runtime

In [28]:
model.profile

{'name': 'Llama 3.1 8B Instant',
 'release_date': '2024-07-23',
 'last_updated': '2024-07-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 131072,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': False,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

In [29]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

Same can be done with TypeDict and Pydantic too